# 12.9 音訊怎麼進入既有模型？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一個圖片佔位格，展開成三個向量**

展開 labels 同步加 ignored，再做下一字 shift。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/modal_expand.svg")))

**先想一想：**聲音加倍長，回答label索引應不變嗎？

聲音projector把Da轉成文字D，音訊placeholder展開成多個時間features。和圖片共享labels契約，但長度取決於聲音時長。

**把直覺寫成數學：**增加長度=T_audio−1；首回答位置依新序列覈對，不預設固定視覺patch數。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.multimodal import MultiModalLM, tone

model = MultiModalLM(TinyLM(ModelConfig(width=8)))
ids = torch.tensor([1, 3, 6, 4, 100, 2])
labels = torch.tensor([-100, -100, -100, -100, 100, 2])
result = model(ids, labels, waveform=tone())
print(result["logits"].shape, result["labels"].shape)
assert result["labels"][0, -2] == 100

**如何讀結果：**音訊features不是UTF-8位元組；模態位置只做context，不當成文字答案label。

**只改一件事：**只增加音訊時長，覈對新的labels長度。
